In [ ]:
#Install Dependencies   
%pip install anthropic python-dotenv

In [ ]:
# Laod Env Variables    
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Create API Client
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-6"  # or "claude-sonnet-4-0-20240924" for the latest version

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    
    # This approach handles an important detail: Claude's API doesn't accept system=None, so you need to conditionally include the system parameter only when it's provided.
    
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,   
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text



In [ ]:
# Putting it all together
# Start with an empty message list
messages = []
print(messages)
# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)
print(answer)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)
print(final_answer)
print(messages)

In [ ]:
while True:
    user_input = input("You: ")
    print (">", user_input )
    # Add user unput to conversation history
    add_user_message(messages, user_input)
    # Get response from Claude with full conversation context
    answer = chat(messages)   
    # Add Claude's response to conversation history
    add_assistant_message(messages, answer)
    print ("-----")
     # Print and add Claude's response to conversation history
    print(f"Claude: {answer}")


In [ ]:
messages = []
# Ensure there's at least one message before calling the API
if not messages:
	add_user_message(messages, "Hello")  # seed the conversation

# Without system prompt
answer = chat(messages)
print(answer)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)
print(answer)



In [ ]:
messages = []

add_user_message(messages, "Write a Python function to check duplicate characters in a string",)
# answer = chat(messages)
answer = chat(messages, system="You are a python engineer who writes very consise code.")
print(answer)
answer

In [ ]:
def chatWithTemperature(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
messages1 = []
add_user_message(messages1, "Write an idea to generate a movie plot")
# Low temperature - more predictable
answer = chatWithTemperature(messages1, temperature=0.0)
 
# High temperature - more creative  
answer = chatWithTemperature(messages1, temperature=1.0)
 

In [ ]:
answer


In [ ]:
# To enable streaming, add stream=True to your messages.create call:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

In [ ]:
# Simplified Text Streaming

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

In [ ]:
# Getting the Complete Message

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client 
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()
    
    print (final_message.content[0].text)
    